In [ ]:
import sys
sys.path.append("../../glass/src")

In [ ]:
import glass.data
import glob
import json

latexdir = "./"
datadir = "groundunitdata/"
samdatadir = "../samdata/"

In [ ]:
for unitfilename in glob.glob("*platoon.json", root_dir="../groundunitdata/") + glob.glob(
    "*battery.json", root_dir="../groundunitdata/"
):

    basefilename = unitfilename[:-12]

    print("reading \"%s\"." % unitfilename)
    data = glass.data.loaddatafile("groundunitdata", unitfilename[:-5])

    # Don't make sections or squads for SAM or radar units.
    if "CCU" in data["name"] or "sam" in data or "radar" in data["symbols"]:
        continue

    print("  making \"%ssection.json\"." % basefilename)

    for member in ["name", "description", "symbols"]:
        data[member] = data[member].replace("platoon", "section")
        data[member] = data[member].replace("battery", "section")

    if "VPs" in data:
        data["VPs"] = data["VPs"][1:]

    if "aaa" in data:
        if data["aaa"]["class"] == "B":
            del data["aaa"]
        else:
            data["aaa"]["hitroll"] = list(
                max(0, hitroll - 1) for hitroll in data["aaa"]["hitroll"]
            )

    sectionpath = "../groundunitdata/" + basefilename + "section.json"
    with open(sectionpath, "w") as f:
        json.dump(data, f, indent=4)

    data = glass.data.loaddatafile("groundunitdata", unitfilename[:-5])

    # Don't make squads for air defense units.

    if "air-defense" in data["symbols"]:
        continue

    print("  making \"%ssquad.json\"." % basefilename)

    for member in ["name", "description", "symbols"]:
        data[member] = data[member].replace("platoon", "squad")
        data[member] = data[member].replace("battery", "squad")

    if "VPs" in data:
        data["VPs"] = data["VPs"][2:]

    if "aaa" in data:
        if data["aaa"]["class"] == "B":
            del data["aaa"]
        else:
            data["aaa"]["hitroll"] = list(
                max(0, hitroll - 2) for hitroll in data["aaa"]["hitroll"]
            )

    squadpath = "../groundunitdata/" + basefilename + "squad.json"
    with open(squadpath, "w") as f:
        json.dump(data, f, indent=4)